In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [ ]:
# Path(s) to oil concentration files
path_v201 = "/Users/francesco/shared/MEDSLIK_FIXED_SEED_COMPARISON_v201_VS_v300/paria/v201/oil_concentration_1km.nc"
path_v300 = "/Users/francesco/shared/MEDSLIK_FIXED_SEED_COMPARISON_v201_VS_v300/paria/v300/oil_concentration_1km.nc"

In [ ]:
# Compute distance from the spill location
spill_lon = -63.55
spill_lat = 10.811

In [ ]:
# Open dataset(s)
oil_v201 = xr.open_dataset(path_v201)
oil_v300 = xr.open_dataset(path_v300)
# Load dataset(s)
oil_v201.load()
oil_v300.load()

In [ ]:
# Transpose dataset(s)
oil_v201 = oil_v201.transpose("lat", "lon", "time")
oil_v300 = oil_v300.transpose("lat", "lon", "time")

In [ ]:
# Define function for computing Root Mean Squared Error (RMSE)
def rmse(arr1: np.ndarray, arr2: np.ndarray) -> np.ndarray:
    """
    Compute Root Mean Squared Error (RMSE) between two 3D arrays (lon,lat,time).
    """
    squared_error = (arr2 - arr1) ** 2
    mean_squared_error = np.nanmean(squared_error, axis=2)
    root_mean_squared_error = np.sqrt(mean_squared_error)
    return root_mean_squared_error

In [ ]:
# Conversion from kg/m^2 to tons/km^2
oil_v201.concentration.values *= 1000
oil_v300.concentration.values *= 1000

In [ ]:
# Compute RMSE
rmse_val = rmse(oil_v201.concentration, oil_v300.concentration)

In [ ]:
# Print RMSE maximum value
rmse_max = np.nanmax(rmse_val)
print(f"RMSE MAX value is {rmse_max}")

In [ ]:
# Plot RMSE
plt.figure(1)
plt.imshow(rmse_val[::-1, :], cmap="jet", vmax=rmse_max)
plt.title("RMSE - LEBANON")
cbar = plt.colorbar()
cbar.set_label("tons/km^2", fontsize=12)
plt.show()
plt.close()

In [ ]:
# If RMSE netcdf output should be created
create_output = False
# Path where the netcdf output file should be saved
output_path_for_rmse_file = ""
if create_output:
    os.makedirs(output_path_for_rmse_file, exist_ok=True)

In [ ]:
if create_output:
    rmse_ds = oil_v201.copy()  # Ensure this creates a copy of the dataset
    # Check shape compatibility
    if rmse_val.shape != (rmse_ds.dims["lat"], rmse_ds.dims["lon"]):
        raise ValueError(
            "Shape of 'rmse' does not match the (lat, lon) dimensions of the dataset."
        )

    rmse_copy = rmse_val.copy()
    rmse_copy[np.where(np.isnan(rmse_copy))] = 0.0
    # Assign the RMSE values for all timesteps
    for t in range(len(rmse_ds.time)):
        rmse_ds["concentration"][:, :, t] = rmse_copy / 1000  # conversion to km/m^2

    # Add an attribute to the DataArray
    rmse_ds["concentration"].attrs[
        "DESCRIPTION"
    ] = "The concentration values correspond to the RMSE"

    # Verify the maximum values
    assert np.max(rmse_ds.concentration.values) == np.max(rmse_val / 1000)

    # Transpose and save the dataset to a NetCDF file
    rmse_ds = rmse_ds.transpose("time", "lat", "lon")
    # Create netcdf output
    filename = "rmse.nc"
    out_path = os.path.join(output_path_for_rmse_file, filename)
    rmse_ds.to_netcdf(out_path)

In [ ]:
# Haversine formula for distances computation
def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """
    # convert decimal degrees to radians
    lon1 = np.radians(lon1)
    lon2 = np.radians(lon2)
    lat1 = np.radians(lat1)
    lat2 = np.radians(lat2)
    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    # 6367 km is the radius of the Earth
    km = 6371 * c
    return km

In [ ]:
# Gravity center coordinates
gc_lon_v201 = oil_v201.lon_gravity_center.values
gc_lat_v201 = oil_v201.lat_gravity_center.values
gc_lon_v300 = oil_v300.lon_gravity_center.values
gc_lat_v300 = oil_v300.lat_gravity_center.values
n_steps = len(gc_lat_v201)
distance_from_spill_position_v201 = np.array(
    [
        haversine(spill_lon, spill_lat, gc_lon_v201[i], gc_lat_v201[i])
        for i in range(n_steps)
    ]
)
distance_from_spill_position_v300 = np.array(
    [
        haversine(spill_lon, spill_lat, gc_lon_v300[i], gc_lat_v300[i])
        for i in range(n_steps)
    ]
)
distance_error = np.abs(distance_from_spill_position_v201 - distance_from_spill_position_v300)/distance_from_spill_position_v201
rmse_distance = np.sqrt(np.mean((distance_from_spill_position_v201 - distance_from_spill_position_v300)**2, axis=0))

In [ ]:
# Plot Distance Center of Gravity
plt.figure(2)
time = np.arange(1, n_steps+1)
plt.plot(time, distance_from_spill_position_v201, c="b")
plt.plot(time, distance_from_spill_position_v300, c="r")
plt.title(f"Oil Slick Gravity Center - distance travelled over time")
plt.text(len(time)*2/3,5,f"RMSE = {rmse_distance:.2f} km", fontweight="bold")
plt.xlabel("time from release (hr)")
plt.ylabel("distance travelled (km)")
plt.legend(["v2.01", "v3.00"])
plt.show()
plt.close()

In [ ]:
# Plot Error of slick gravity center position
plt.figure(3)
time = np.arange(1, n_steps+1)
plt.plot(time, distance_error*100, c="k")
plt.title("Oil Slick Gravity Center - distance travelled over time \n Error Relative to Distance Travelled")
plt.xlabel("time from release (hr)")
plt.ylabel("%")
plt.show()
plt.close()